In [166]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/brain_tumor_dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Patient_ID           20000 non-null  int64  
 1   Age                  20000 non-null  int64  
 2   Gender               20000 non-null  object 
 3   Tumor_Type           20000 non-null  object 
 4   Tumor_Size           20000 non-null  float64
 5   Location             20000 non-null  object 
 6   Histology            20000 non-null  object 
 7   Stage                20000 non-null  object 
 8   Symptom_1            20000 non-null  object 
 9   Symptom_2            20000 non-null  object 
 10  Symptom_3            20000 non-null  object 
 11  Radiation_Treatment  20000 non-null  object 
 12  Surgery_Performed    20000 non-null  object 
 13  Chemotherapy         20000 non-null  object 
 14  Survival_Rate        20000 non-null  float64
 15  Tumor_Growth_Rate    20000 non-null 

In [167]:
df.head()

,Patient_ID,Age,Gender,Tumor_Type,Tumor_Size,Location,Histology,Stage,Symptom_1,Symptom_2,Symptom_3,Radiation_Treatment,Surgery_Performed,Chemotherapy,Survival_Rate,Tumor_Growth_Rate,Family_History,MRI_Result,Follow_Up_Required
0,1,73,Male,Malignant,5.375612,Temporal,Astrocytoma,III,Vision Issues,Seizures,Seizures,No,No,No,51.312579,0.111876,No,Positive,Yes
1,2,26,Male,Benign,4.847098,Parietal,Glioblastoma,II,Headache,Headache,Nausea,Yes,Yes,Yes,46.373273,2.165736,Yes,Positive,Yes
2,3,31,Male,Benign,5.588391,Parietal,Meningioma,I,Vision Issues,Headache,Seizures,No,No,No,47.072221,1.884228,No,Negative,No
3,4,29,Male,Malignant,1.436600,Temporal,Medulloblastoma,IV,Vision Issues,Seizures,Headache,Yes,No,Yes,51.853634,1.283342,Yes,Negative,No
4,5,54,Female,Benign,2.417506,Parietal,Glioblastoma,I,Headache,Headache,Seizures,No,No,Yes,54.708987,2.069477,No,Positive,Yes


In [168]:
df.drop(columns=["Patient_ID"], inplace=True)

In [169]:
df.duplicated().sum()

np.int64(0)

### Encode

In [170]:
def label_encode(df: pd.DataFrame):
    struct = dict()
    for col in df.select_dtypes("object").columns.to_list():
        unq = df[col].unique()
        unq_map = {v: k for k, v in enumerate(unq)}
        struct[col] = unq_map
        df[col] = df[col].map(unq_map)
    
    return df, struct

In [171]:
df_enc, struct = label_encode(df)
df_enc.head()

,Age,Gender,Tumor_Type,Tumor_Size,Location,Histology,Stage,Symptom_1,Symptom_2,Symptom_3,Radiation_Treatment,Surgery_Performed,Chemotherapy,Survival_Rate,Tumor_Growth_Rate,Family_History,MRI_Result,Follow_Up_Required
0,73,0,0,5.375612,0,0,0,0,0,0,0,0,0,51.312579,0.111876,0,0,0
1,26,0,1,4.847098,1,1,1,1,1,1,1,1,1,46.373273,2.165736,1,0,0
2,31,0,1,5.588391,1,2,2,0,1,0,0,0,0,47.072221,1.884228,0,1,1
3,29,0,0,1.436600,0,3,3,0,0,2,1,0,1,51.853634,1.283342,1,1,1
4,54,1,1,2.417506,1,1,2,1,1,0,0,0,1,54.708987,2.069477,0,0,0


In [172]:
struct

{'Gender': {'Male': 0, 'Female': 1},
 'Tumor_Type': {'Malignant': 0, 'Benign': 1},
 'Location': {'Temporal': 0, 'Parietal': 1, 'Frontal': 2, 'Occipital': 3},
 'Histology': {'Astrocytoma': 0,
  'Glioblastoma': 1,
  'Meningioma': 2,
  'Medulloblastoma': 3},
 'Stage': {'III': 0, 'II': 1, 'I': 2, 'IV': 3},
 'Symptom_1': {'Vision Issues': 0, 'Headache': 1, 'Seizures': 2, 'Nausea': 3},
 'Symptom_2': {'Seizures': 0, 'Headache': 1, 'Vision Issues': 2, 'Nausea': 3},
 'Symptom_3': {'Seizures': 0, 'Nausea': 1, 'Headache': 2, 'Vision Issues': 3},
 'Radiation_Treatment': {'No': 0, 'Yes': 1},
 'Surgery_Performed': {'No': 0, 'Yes': 1},
 'Chemotherapy': {'No': 0, 'Yes': 1},
 'Family_History': {'No': 0, 'Yes': 1},
 'MRI_Result': {'Positive': 0, 'Negative': 1},
 'Follow_Up_Required': {'Yes': 0, 'No': 1}}

### Split

In [173]:
def train_test_split(X: pd.DataFrame, y: pd.DataFrame, test_size=0.2, random_state=None):
    if random_state:
        np.random.seed(random_state)
    
    n_samples = len(X)
    shuffle_indices = np.random.permutation(np.arange(n_samples))

    test_size = round(n_samples * test_size)
    train_size = test_size - n_samples

    train_indices = shuffle_indices[:train_size]
    test_indices = shuffle_indices[test_size:]

    X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
    y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]

    return [X_train, X_test, y_train, y_test]

In [174]:
X = df.drop(columns=["Tumor_Type"])
y = df.Tumor_Type

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

X_train.head()

,Age,Gender,Tumor_Size,Location,Histology,Stage,Symptom_1,Symptom_2,Symptom_3,Radiation_Treatment,Surgery_Performed,Chemotherapy,Survival_Rate,Tumor_Growth_Rate,Family_History,MRI_Result,Follow_Up_Required
10650,60,0,4.185355,0,2,1,3,1,1,0,0,1,94.739701,0.128492,0,0,0
2041,29,1,6.391747,0,2,2,3,2,0,1,0,1,49.751461,1.724219,1,0,1
8668,28,0,1.129997,1,1,3,0,0,3,1,1,1,95.452798,1.228702,1,0,0
1114,69,1,2.188328,0,0,1,0,1,2,1,0,0,92.178737,1.096059,0,1,0
13902,69,1,5.847122,2,0,1,1,3,0,1,0,0,90.654556,2.415105,0,1,0


### Scaler

In [175]:
def min_max_scaler(X: pd.DataFrame):
    return (X - X.min(axis=0) / (X.max(axis=0) - X.min(axis=0)))

def std_scaler(X: pd.DataFrame):
    return (X - X.mean(axis=0) / X.std(axis=0))

In [176]:
X_train_mm_scaler = min_max_scaler(X_train)
X_test_mm_scaler = min_max_scaler(X_test)

X_train_mm_scaler.head()

,Age,Gender,Tumor_Size,Location,Histology,Stage,Symptom_1,Symptom_2,Symptom_3,Radiation_Treatment,Surgery_Performed,Chemotherapy,Survival_Rate,Tumor_Growth_Rate,Family_History,MRI_Result,Follow_Up_Required
10650,59.661017,0.0,4.132303,0.0,2.0,1.0,3.0,1.0,1.0,0.0,0.0,1.0,94.072948,0.093033,0.0,0.0,0.0
2041,28.661017,1.0,6.338695,0.0,2.0,2.0,3.0,2.0,0.0,1.0,0.0,1.0,49.084707,1.688760,1.0,0.0,1.0
8668,27.661017,0.0,1.076944,1.0,1.0,3.0,0.0,0.0,3.0,1.0,1.0,1.0,94.786044,1.193243,1.0,0.0,0.0
1114,68.661017,1.0,2.135275,0.0,0.0,1.0,0.0,1.0,2.0,1.0,0.0,0.0,91.511983,1.060600,0.0,1.0,0.0
13902,68.661017,1.0,5.794070,2.0,0.0,1.0,1.0,3.0,0.0,1.0,0.0,0.0,89.987803,2.379645,0.0,1.0,0.0


In [177]:
X_train_std_scaler = std_scaler(X_train)
X_test_std_scaler = std_scaler(X_test)

X_train_std_scaler.head()

,Age,Gender,Tumor_Size,Location,Histology,Stage,Symptom_1,Symptom_2,Symptom_3,Radiation_Treatment,Surgery_Performed,Chemotherapy,Survival_Rate,Tumor_Growth_Rate,Family_History,MRI_Result,Follow_Up_Required
10650,57.107867,-1.017529,2.269058,-1.337359,0.662764,-0.332317,1.656725,-0.327769,-0.364645,-0.987454,-1.004385,0.006106,90.646669,-1.737122,-0.998376,-1.007402,-0.98696
2041,26.107867,-0.017529,4.475450,-1.337359,0.662764,0.667683,1.656725,0.672231,-1.364645,0.012546,-1.004385,0.006106,45.658428,-0.141395,0.001624,-1.007402,0.01304
8668,25.107867,-1.017529,-0.786300,-0.337359,-0.337236,1.667683,-1.343275,-1.327769,1.635355,0.012546,-0.004385,0.006106,91.359766,-0.636912,0.001624,-1.007402,-0.98696
1114,66.107867,-0.017529,0.272031,-1.337359,-1.337236,-0.332317,-1.343275,-0.327769,0.635355,0.012546,-1.004385,-0.993894,88.085705,-0.769556,-0.998376,-0.007402,-0.98696
13902,66.107867,-0.017529,3.930825,0.662641,-1.337236,-0.332317,-0.343275,1.672231,-1.364645,0.012546,-1.004385,-0.993894,86.561524,0.549490,-0.998376,-0.007402,-0.98696


### Build Model

In [178]:
class KNN:
    def __init__(self, k:int, p:int, weight, random_state=None):
        self.k = k
        self.p = p
        self.weight = weight
        self.random_state = random_state
    
    def _minkowski_distance(self, a: pd.DataFrame, b: pd.DataFrame):
        return np.power(np.sum(np.abs(a - b) ** self.p), 1/self.p)
    
    def _get_nn(self, X_test: pd.DataFrame):
        if self.random_state:
            np.random.seed(self.random_state)

        distance = [self._minkowski_distance(x, self.X_train) for x in X_test]
        k_idxs = np.argsort(distance)[:self.k]
        k_labels = [self.y_train[i] for i in k_idxs]
        k_distance = [distance[i] for i in k_idxs]

        if self.weight == "uniform":
            values, counts = np.unique(k_labels, return_counts=True)
            max_count = np.max(counts)
            candidates = values[counts == max_count]
            return np.random.choice(candidates)
        
        elif self.weight == "distance":
            weights = [1 / (d + 1e-5) for d in k_distance]

        elif callable(self.weight):
            weights = [self.weight(d) for d in k_distance]
        
        label_weight = dict()
        for label, weight in zip(k_labels, weights):
            label_weight[label] = label_weight.get(label, 0) + weight
        
        max_weight = max(label_weight.values())
        candidates = [
            label 
            for label, weight in label_weight.items() 
            if weight == max_weight
        ]
        return np.random.choice(candidates)

    def fit(self, X_train: pd.DataFrame, y_train: pd.DataFrame):
        self.X_train = X_train.to_numpy()
        self.y_train = y_train.to_numpy()

    def predict(self, X_test: pd.DataFrame):
        return np.array([
            self._get_nn(x)
            for x in X_test.to_numpy()
        ])

In [179]:
class EvaluationModel:
    def __init__(self, y_true: pd.DataFrame, y_pred: np.ndarray):
        self.y_true = y_true.to_numpy()
        self.y_pred = y_pred
    
    def accuracy(self):
        return np.sum(self.y_true == self.y_pred) / len(self.y_true)
    
    def precision(self, TP, FP):
        return TP / (TP + FP) if (TP + FP) != 0 else 0
    
    def recall(self, TP, FN):
        return TP / (TP + FN) if (TP + FN) != 0 else 0
    
    def f1_score(self, TP, FP, FN):
        P = self.precision(TP, FP)
        R = self.recall(TP, FN)

        return 2 * (P * R) / (P + R) if (P + R) != 0 else 0
    
    def macro_avg(self, report_df: pd.DataFrame):
        return report_df[["Precision", "Recall", "F1-Score"]].mean()
    
    def weighted_avg(self, report_df: pd.DataFrame):
        weights = np.bincount(self.y_true)
        total = weights.sum()

        return (report_df[["Precision", "Recall", "F1-Score"]].T * weights / total).T.sum()

    def make_report(self):
        cats = pd.Series(self.y_true).unique()

        reports = {"Category": [], "Precision": [], "Recall": [], "F1-Score": []}

        for cat in cats:
            # Confusion Matrixs
            TP = np.sum((self.y_true == cat) & (self.y_pred == cat)) # Asli positif prediksi positif
            TN = np.sum((self.y_true != cat) & (self.y_pred != cat)) # Asli negatif prediksi negatif
            FP = np.sum((self.y_true != cat) & (self.y_pred == cat)) # Asli negatif prediksi positif
            FN = np.sum((self.y_true == cat) & (self.y_pred != cat)) # Asli positif prediksi negatif

            # Calculation Matrixs
            P = self.precision(TP, FP)
            R = self.recall(TP, FN)
            F1 = self.f1_score(TP, FP, FN)

            # Append Matrixs
            reports["Category"].append(cat)
            reports["Precision"].append(P)
            reports["Recall"].append(R)
            reports["F1-Score"].append(F1)

        accuracy = self.accuracy()
        result_df = pd.DataFrame(reports)
        macro_avg = self.macro_avg(result_df)
        weighted_avg = self.macro_avg(result_df)

        summary_record = pd.DataFrame.from_records([
            {"Category": 'macro avg', **macro_avg},
            {"Category": 'weighted avg', **weighted_avg},
        ])

        result_df = pd.concat([result_df, summary_record], ignore_index=True)

        return result_df, accuracy

In [180]:
def log_model_process(params, index, total):
    progress_bar = f"[{index}/{total}] [{index/total*100:.1f}%]"
    separator = "=" * 100
    
    print(f"{separator}")
    print(f"{progress_bar} Evaluating model with params:")
    print(params)
    print(f"{separator}\n")

def search_best_model(params: dict, limit_data=None):
    total = (
        len(params["K"]) *
        len(params["P"]) *
        len(params["weights"]) *
        len(params["random_states"]) *
        len(params["scaler"])
    )

    count = 1

    models = list()

    for k in params["K"]:
        for p in params["P"]:
            for weight in params["weights"]:
                for rs in params["random_states"]:
                    for i, (train, test) in enumerate(params["scaler"]):
                        # Initialization model
                        model = KNN(k=k, p=p, weight=weight, random_state=rs)
                        model.fit(X_train=train[:limit_data], y_train=y_train[:limit_data])
                        
                        preds = model.predict(X_test=test[:limit_data])

                        # Evalutaion
                        eval_model = EvaluationModel(y_pred=preds, y_true=y_test[:limit_data])
                        result_df, accuracy = eval_model.make_report()

                        params_result = {
                            "K": k,
                            "P": p,
                            "weights": weight,
                            "random_states": rs,
                            "scaler": i
                        }

                        log_model_process(params_result, count, total)

                        models.append({
                            "params": params_result,
                            "accuracy": accuracy,
                            "result_df": result_df
                        })

                        count += 1
    return models

In [181]:
def custom_distance(d):
    return 1 / (d ** 2 + 1e-5)

params = {
    "K": range(3, 9),
    "P": range(1, 3),
    "weights": ["uniform", "distance", custom_distance],
    "random_states": [0, 42, 99],
    "scaler": [(X_train_mm_scaler, X_test_mm_scaler), (X_train_std_scaler, X_test_std_scaler), (X_train, X_test)]
}

models = search_best_model(params, limit_data=100)

[1/324] [0.3%] Evaluating model with params:
{'K': 3, 'P': 1, 'weights': 'uniform', 'random_states': 0, 'scaler': 0}

[2/324] [0.6%] Evaluating model with params:
{'K': 3, 'P': 1, 'weights': 'uniform', 'random_states': 0, 'scaler': 1}

[3/324] [0.9%] Evaluating model with params:
{'K': 3, 'P': 1, 'weights': 'uniform', 'random_states': 0, 'scaler': 2}

[4/324] [1.2%] Evaluating model with params:
{'K': 3, 'P': 1, 'weights': 'uniform', 'random_states': 42, 'scaler': 0}

[5/324] [1.5%] Evaluating model with params:
{'K': 3, 'P': 1, 'weights': 'uniform', 'random_states': 42, 'scaler': 1}

[6/324] [1.9%] Evaluating model with params:
{'K': 3, 'P': 1, 'weights': 'uniform', 'random_states': 42, 'scaler': 2}

[7/324] [2.2%] Evaluating model with params:
{'K': 3, 'P': 1, 'weights': 'uniform', 'random_states': 99, 'scaler': 0}

[8/324] [2.5%] Evaluating model with params:
{'K': 3, 'P': 1, 'weights': 'uniform', 'random_states': 99, 'scaler': 1}

[9/324] [2.8%] Evaluating model with params:
{'K': 

In [182]:
sorted_models = sorted(models, key=lambda x: x["accuracy"], reverse=True)[:3]

for model in sorted_models:
    print(f"Params: {model["params"]}")
    display(model["result_df"])
    print(f"Accuracy: {model["accuracy"]}\n\n")

Params: {'K': 4, 'P': 2, 'weights': 'uniform', 'random_states': 99, 'scaler': 1}


,Category,Precision,Recall,F1-Score
0,1,0.631579,0.272727,0.380952
1,0,0.604938,0.875000,0.715328
2,macro avg,0.618259,0.573864,0.548140
3,weighted avg,0.618259,0.573864,0.548140


Accuracy: 0.61


Params: {'K': 4, 'P': 2, 'weights': 'uniform', 'random_states': 0, 'scaler': 0}


,Category,Precision,Recall,F1-Score
0,1,0.750000,0.136364,0.230769
1,0,0.586957,0.964286,0.729730
2,macro avg,0.668478,0.550325,0.480249
3,weighted avg,0.668478,0.550325,0.480249


Accuracy: 0.6


Params: {'K': 3, 'P': 2, 'weights': 'uniform', 'random_states': 0, 'scaler': 1}


,Category,Precision,Recall,F1-Score
0,1,0.714286,0.113636,0.196078
1,0,0.580645,0.964286,0.724832
2,macro avg,0.647465,0.538961,0.460455
3,weighted avg,0.647465,0.538961,0.460455


Accuracy: 0.59


